In [1]:
!pip install gensim
!pip install nltk
!pip install scikit-learn
!pip install matplotlib

  Using cached smart_open-7.5.0-py3-none-any.whl.metadata (24 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 18.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 39.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.3/30.3 MB 55.3 MB/s  0:00:00 eta 0:00:01
Using cached smart_open-7.5.0-py3-none-any.whl (63 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [gensim]2m4/5 [gensim]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
  Using cached nltk-3.9.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached nltk-3.9.2-py3-none-any.whl (1.5 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [nltk]4/5 [nltk]b]

[notice] A new release of pip is available

Word Embedding Techniques with Gensim: Complete Walkthrough
1. Setup and Data Preparation

In [2]:
# Install required packages
# pip install gensim nltk matplotlib scikit-learn

import gensim
from gensim.models import Word2Vec, FastText, KeyedVectors
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.models import LdaModel
from gensim.corpora import Dictionary
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import logging
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
nltk.download('wordnet')
nltk.download('omw-1.4')
lemmatizer = WordNetLemmatizer()
# Set up logging to see training progress
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
# Sample corpus for demonstration
sample_texts = [
    "The cat sat on the mat and looked out the window",
    "Dogs are loyal animals and great companions for humans",
    "Machine learning algorithms can process large amounts of data",
    "Natural language processing helps computers understand human language",
    "Deep learning models require significant computational resources",
    "The weather today is sunny and warm with clear skies",
    "Books contain knowledge and stories that educate and entertain",
    "Music has the power to evoke emotions and memories",
    "Technology continues to advance at an unprecedented pace",
    "Cooking delicious meals brings people together around the table"
]

# Preprocess the corpus
def preprocess_text(texts):
    processed_texts = []
    for text in texts:
        # Tokenize and convert to lowercase
        tokens = word_tokenize(text.lower())
        # Remove punctuation and keep only alphabetic tokens
        tokens = [token for token in tokens if token.isalpha()]
        
        #lemmatisation 
        tokens = [lemmatizer.lemmatize(token) for token in tokens]

        #filter stop words
        filtered_tokens = [token for token in tokens if token not in stop_words]

        processed_texts.append(filtered_tokens)

    return processed_texts

processed_corpus = preprocess_text(sample_texts)
print("Preprocessed corpus sample:")
for i, doc in enumerate(processed_corpus[:3]):
    print(f"Document {i+1}: {doc}")


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/sureshveluru/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/sureshveluru/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/sureshveluru/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sureshveluru/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Preprocessed corpus sample:
Document 1: ['cat', 'sat', 'mat', 'looked', 'window']
Document 2: ['dog', 'loyal', 'animal', 'great', 'companion', 'human']
Document 3: ['machine', 'learning', 'algorithm', 'process', 'large', 'amount', 'data']


2. Word2Vec Implementation
Word2Vec creates dense vector representations by predicting context words (Skip-gram) or target words from context (CBOW).

In [3]:
# Train Word2Vec model
print("\n=== WORD2VEC TRAINING ===")

# CBOW model (predicts target word from context)
w2v_cbow = Word2Vec(
    sentences=processed_corpus,
    vector_size=100,        # Dimensionality of word vectors
    window=5,              # Maximum distance between target and context words
    min_count=1,           # Ignore words with frequency less than this
    workers=4,             # Number of threads
    sg=0,                  # 0 for CBOW, 1 for Skip-gram
    epochs=100
)

# Skip-gram model (predicts context words from target)
w2v_skipgram = Word2Vec(
    sentences=processed_corpus,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,                  # Skip-gram
    epochs=100
)

print(f"Vocabulary size: {len(w2v_cbow.wv.key_to_index)}")
print(f"Vector dimensions: {w2v_cbow.wv.vector_size}")

# Get word vector
word_vector = w2v_cbow.wv['computer']
print(f"Vector for 'computer': {word_vector[:10]}...")  # Show first 10 dimensions

# Find similar words
similar_words = w2v_cbow.wv.most_similar('computer', topn=5)
print(f"Words similar to 'computer': {similar_words}")

# Word analogy (king - man + woman = queen concept)
try:
    analogy = w2v_cbow.wv.most_similar(positive=['learning', 'data'], negative=['language'], topn=3)
    print(f"Analogy (learning + data - language): {analogy}")
except KeyError as e:
    print(f"Analogy failed: {e}")


2026-02-12 23:38:29,188 : INFO : collecting all words and their counts
2026-02-12 23:38:29,188 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-02-12 23:38:29,189 : INFO : collected 61 word types from a corpus of 64 raw words and 10 sentences
2026-02-12 23:38:29,189 : INFO : Creating a fresh vocabulary
2026-02-12 23:38:29,190 : INFO : Word2Vec lifecycle event {'msg': 'effective_min_count=1 retains 61 unique words (100.00% of original 61, drops 0)', 'datetime': '2026-02-12T23:38:29.190649', 'gensim': '4.4.0', 'python': '3.9.24 (main, Oct  9 2025, 11:54:19) \n[Clang 17.0.0 (clang-1700.3.19.1)]', 'platform': 'macOS-26.1-arm64-arm-64bit', 'event': 'prepare_vocab'}
2026-02-12 23:38:29,191 : INFO : Word2Vec lifecycle event {'msg': 'effective_min_count=1 leaves 64 word corpus (100.00% of original 64, drops 0)', 'datetime': '2026-02-12T23:38:29.191003', 'gensim': '4.4.0', 'python': '3.9.24 (main, Oct  9 2025, 11:54:19) \n[Clang 17.0.0 (clang-1700.3.19.1)]', 'plat


=== WORD2VEC TRAINING ===
Vocabulary size: 61
Vector dimensions: 100
Vector for 'computer': [-0.00650965  0.00893761 -0.00184185 -0.00832171  0.00815381 -0.00349762
 -0.00106466  0.00015518  0.0027357  -0.00541168]...
Words similar to 'computer': [('human', 0.23879525065422058), ('story', 0.1918739229440689), ('sky', 0.15110714733600616), ('resource', 0.13815900683403015), ('emotion', 0.1375775784254074)]
Analogy (learning + data - language): [('educate', 0.1879795491695404), ('understand', 0.18728183209896088), ('table', 0.18156658113002777)]


3. FastText Implementation
FastText extends Word2Vec by considering subword information, making it better at handling rare words and morphological variations.

In [21]:
print("\n=== FASTTEXT TRAINING ===")

# Train FastText model
fasttext_model = FastText(
    sentences=processed_corpus,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,                  # Skip-gram
    min_n=3,              # Minimum character n-gram length
    max_n=6,              # Maximum character n-gram length
    epochs=100
)

print(f"FastText vocabulary size: {len(fasttext_model.wv.key_to_index)}")

# FastText can handle out-of-vocabulary words
oov_word = "computering"  # Not in training data
try:
    oov_vector = fasttext_model.wv[oov_word]
    print(f"Vector for OOV word '{oov_word}': {oov_vector[:5]}...")
    
    # Find similar words for OOV
    similar_oov = fasttext_model.wv.most_similar(oov_word, topn=3)
    print(f"Words similar to OOV '{oov_word}': {similar_oov}")
except KeyError:
    print(f"Could not generate vector for '{oov_word}'")

# Compare with regular word
regular_similar = fasttext_model.wv.most_similar('learning', topn=3)
print(f"Words similar to 'learning': {regular_similar}")


2026-02-14 22:46:48,643 : INFO : collecting all words and their counts
2026-02-14 22:46:48,645 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-02-14 22:46:48,645 : INFO : collected 8 word types from a corpus of 8 raw words and 2 sentences
2026-02-14 22:46:48,645 : INFO : Creating a fresh vocabulary
2026-02-14 22:46:48,648 : INFO : FastText lifecycle event {'msg': 'effective_min_count=1 retains 8 unique words (100.00% of original 8, drops 0)', 'datetime': '2026-02-14T22:46:48.647917', 'gensim': '4.4.0', 'python': '3.9.24 (main, Oct  9 2025, 11:54:19) \n[Clang 17.0.0 (clang-1700.3.19.1)]', 'platform': 'macOS-26.1-arm64-arm-64bit', 'event': 'prepare_vocab'}
2026-02-14 22:46:48,648 : INFO : FastText lifecycle event {'msg': 'effective_min_count=1 leaves 8 word corpus (100.00% of original 8, drops 0)', 'datetime': '2026-02-14T22:46:48.648953', 'gensim': '4.4.0', 'python': '3.9.24 (main, Oct  9 2025, 11:54:19) \n[Clang 17.0.0 (clang-1700.3.19.1)]', 'platform': 


=== FASTTEXT TRAINING ===


2026-02-14 22:46:49,024 : INFO : FastText lifecycle event {'update': False, 'trim_rule': 'None', 'datetime': '2026-02-14T22:46:49.024305', 'gensim': '4.4.0', 'python': '3.9.24 (main, Oct  9 2025, 11:54:19) \n[Clang 17.0.0 (clang-1700.3.19.1)]', 'platform': 'macOS-26.1-arm64-arm-64bit', 'event': 'build_vocab'}
2026-02-14 22:46:49,024 : INFO : FastText lifecycle event {'msg': 'training model with 4 workers on 8 vocabulary and 100 features, using sg=1 hs=0 sample=0.001 negative=5 window=5 shrink_windows=True', 'datetime': '2026-02-14T22:46:49.024692', 'gensim': '4.4.0', 'python': '3.9.24 (main, Oct  9 2025, 11:54:19) \n[Clang 17.0.0 (clang-1700.3.19.1)]', 'platform': 'macOS-26.1-arm64-arm-64bit', 'event': 'train'}
2026-02-14 22:46:49,031 : INFO : EPOCH 0: training on 8 raw words (1 effective words) took 0.0s, 28777 effective words/s
2026-02-14 22:46:49,032 : INFO : EPOCH 1: training on 8 raw words (0 effective words) took 0.0s, 0 effective words/s
2026-02-14 22:46:49,032 : INFO : EPOCH 2:

FastText vocabulary size: 8
Vector for OOV word 'computering': [ 0.00038094  0.00106995 -0.000363   -0.00044641 -0.00037483]...
Words similar to OOV 'computering': [('learning', 0.25274911522865295), ('embeddings', 0.1733809858560562), ('are', 0.15373966097831726)]
Words similar to 'learning': [('embeddings', 0.17206132411956787), ('useful', 0.06436070054769516), ('machine', -0.0020809669513255358)]


4. Pre-trained Embeddings
Using pre-trained embeddings like GloVe or Google's Word2Vec.

In [24]:
print("\n=== PRE-TRAINED EMBEDDINGS ===")
"""
# Note: You need to download these models first
# For Google's Word2Vec (GoogleNews-vectors-negative300.bin):
# https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit

# For GloVe, convert to Word2Vec format first:
# from gensim.scripts.glove2word2vec import glove2word2vec
# glove2word2vec('glove.6B.100d.txt', 'word2vec_glove.txt')

# Example loading (uncomment if you have the files):
"""
# Load Google's pre-trained Word2Vec
from gensim.models import Word2Vec, FastText, KeyedVectors
google_w2v = KeyedVectors.load_word2vec_format(
    'GoogleNews-vectors-negative300.bin', 
    binary=True
)

# Load converted GloVe vectors
glove_w2v = KeyedVectors.load_word2vec_format(
    'word2vec_glove.txt', 
    binary=False
)

# Use pre-trained embeddings
king_vector = google_w2v['king']
queen_analogy = google_w2v.most_similar(
    positive=['king', 'woman'], 
    negative=['man'], 
    topn=1
)
print(f"King - Man + Woman = {queen_analogy}")


print("To use pre-trained embeddings, download and load appropriate model files")


2026-02-14 22:49:48,122 : INFO : loading projection weights from GoogleNews-vectors-negative300.bin



=== PRE-TRAINED EMBEDDINGS ===


FileNotFoundError: [Errno 2] No such file or directory: 'GoogleNews-vectors-negative300.bin'